# Gold: patient_360

Builds `<catalog>.gold.patient_360` (one row per patient) from silver patients, hospitals, insurance_providers and aggregated lab_results, per the STTM `sttm_silver_to_gold.xlsx`. Full rebuild (CREATE OR REPLACE), LEFT JOINs from patients so no patient is dropped. Insurance providers and hospitals are deduplicated by key before joining because silver currently holds exact duplicate rows.

In [0]:
# Catalog placeholder from the STTM (<catalog>); override via widget or job parameter
dbutils.widgets.text("catalog", "adb_caresync", "Catalog")
print("Using catalog:", dbutils.widgets.get("catalog"))

In [0]:
%sql
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.gold.patient_360')
COMMENT 'Gold: one row per patient with hospital, insurer and lab test summary (STTM silver to gold v1.0)'
AS
WITH hospitals AS (
  -- dedupe defensively: one row per hospital_id
  SELECT hospital_id, hospital_name
  FROM IDENTIFIER(:catalog || '.silver.hospitals')
  QUALIFY row_number() OVER (PARTITION BY hospital_id ORDER BY update_timestamp DESC) = 1
),
insurers AS (
  -- silver.insurance_providers contains exact duplicate rows; keep one per id
  SELECT insurance_provider_id, provider_name
  FROM IDENTIFIER(:catalog || '.silver.insurance_providers')
  QUALIFY row_number() OVER (PARTITION BY insurance_provider_id ORDER BY update_timestamp DESC) = 1
),
lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                                         AS total_tests,
    COUNT_IF(result_status = 'Abnormal')             AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')             AS critical_test_count,
    MAX(test_date)                                   AS last_test_date
  FROM IDENTIFIER(:catalog || '.silver.lab_results')
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                              AS patient_id,
  p.first_name                                                           AS first_name,
  p.last_name                                                            AS last_name,
  CAST(p.date_of_birth AS DATE)                                          AS date_of_birth,
  CAST(floor(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender                                                               AS gender,
  p.city                                                                 AS city,
  p.state                                                                AS state,
  CAST(p.registered_hospital_id AS INT)                                  AS hospital_id,
  h.hospital_name                                                        AS hospital_name,
  CAST(p.insurance_provider_id AS INT)                                   AS insurance_provider_id,
  i.provider_name                                                        AS insurance_provider_name,
  CAST(coalesce(l.total_tests, 0) AS INT)                                AS total_tests,
  CAST(coalesce(l.abnormal_test_count, 0) AS INT)                        AS abnormal_test_count,
  CAST(coalesce(l.critical_test_count, 0) AS INT)                        AS critical_test_count,
  l.last_test_date                                                       AS last_test_date
FROM IDENTIFIER(:catalog || '.silver.patients') p
LEFT JOIN hospitals h ON p.registered_hospital_id = h.hospital_id
LEFT JOIN insurers  i ON p.insurance_provider_id  = i.insurance_provider_id
LEFT JOIN lab_agg   l ON p.patient_id             = l.patient_id

In [0]:
import json

# Count the number of rows processed in this batch (rows written to the gold table)
target_table = f"{dbutils.widgets.get('catalog')}.gold.patient_360"
updated_df = spark.table(target_table)
record_count = updated_df.count()
print(f"{target_table}: {record_count} rows")

# End the notebook and return the record count to the calling job or notebook
dbutils.notebook.exit(json.dumps({"record_count": record_count}))